# Bab 27: Proyek Ujung ke Ujung dengan CRISP-DM

Notebook pendamping buku *Data Mining dengan Python: Konsep, Algoritme, dan Studi Kasus dengan CRISP-DM*. Setiap sel kode sama dengan listing di buku; keluaran yang dicetak di buku ada di berkas `.out` di folder `kode/bab27/`.

Jalankan sel secara berurutan.

In [ ]:
# Jalankan sel ini sekali di awal sesi.
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.exists("kode"):
    if not os.path.exists("data-mining-dengan-python"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/bsnegara/data-mining-dengan-python"])
    os.chdir("data-mining-dengan-python")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "kode/requirements-colab.txt"])
while not os.path.exists("kode/data") and os.getcwd() != "/":
    os.chdir("..")                     # cari akar repositori
sys.path.insert(0, "kode/bab27")
print("Folder kerja:", os.getcwd())

## Listing 27.1: Modul bersama: atribut dan pipeline model

`kode/bab27/siapkan.py`

In [ ]:
"""Modul bersama proyek retensi: atribut, rekayasa fitur, model."""
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA = "kode/data/"
NUMERIK = ["umur", "frekuensi_90h", "frekuensi_90h_lalu",
           "recency_hari", "nilai_90h", "rata_nilai_struk",
           "porsi_promo", "porsi_aplikasi", "porsi_kopi_hitam",
           "porsi_pagi", "porsi_akhir_pekan", "ragam_menu",
           "keluhan_90h", "jarak_km", "pakai_ulasan", "tren",
           "lama_anggota_bln"]
KATEGORIK = ["kota", "tingkat", "jenis_kelamin"]
NILAI_TAHAN, BIAYA = 180_000, 15_000   # Bab 7: nilai, biaya voucer


def fitur(df, potong, kategorik=KATEGORIK):
    """Semua atribut dihitung dari data sampai tanggal potong."""
    df = df.copy()
    df["tren"] = df.frekuensi_90h / (df.frekuensi_90h_lalu + 1)
    df["lama_anggota_bln"] = (pd.Timestamp(potong)
                              - pd.to_datetime(df.tanggal_daftar)
                              ).dt.days / 30.44
    return df[NUMERIK + kategorik]


def buat_model(C=3.162, kategorik=KATEGORIK):
    pra = ColumnTransformer(
        [("num", StandardScaler(), NUMERIK),
         ("kat", OneHotEncoder(handle_unknown="ignore"), kategorik)])
    return make_pipeline(pra, LogisticRegression(C=C, max_iter=3000))

## Listing 27.2: Melatih, memeriksa, dan menyimpan model

`kode/bab27/latih_model.py`

In [ ]:
import sys; sys.modules.pop('siapkan', None); sys.path.insert(0, 'kode/bab27')
import json
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from siapkan import DATA, buat_model, fitur

d = pd.read_csv(DATA + "pelanggan.csv")
X, y = fitur(d, "2026-03-31"), d.churn
print(f"data latih: {len(d)} anggota per 31 Maret 2026, "
      f"churn {y.mean():.3f}")

cv = StratifiedKFold(5, shuffle=True, random_state=0)
p = cross_val_predict(buat_model(), X, y, cv=cv,
                      method="predict_proba")[:, 1]
print(f"AUC-CV {roc_auc_score(y, p):.3f}, Brier "
      f"{brier_score_loss(y, p):.4f}")
desil = pd.qcut(p, 10, labels=False)
kal = pd.DataFrame({"p": p, "y": y}).groupby(desil).mean()
print("kalibrasi per desil (rata-rata tebakan vs nyata):")
print("  " + "  ".join(f"{a:.2f}/{b:.2f}" for a, b in
                       kal.to_numpy()[[0, 3, 6, 8, 9]]))

model = buat_model().fit(X, y)
folder = Path("kode/bab27/model")
folder.mkdir(exist_ok=True)
joblib.dump(model, folder / "churn_v1.joblib")
meta = {"versi": "churn-v1", "dilatih": "2026-04-07",
        "data": "pelanggan.csv (potong 2026-03-31)",
        "auc_cv": round(roc_auc_score(y, p), 4),
        "sklearn": sklearn.__version__,
        "atribut": list(X.columns),
        "skor_acuan": np.quantile(p, np.linspace(0, 1, 11)).tolist()}
(folder / "churn_v1.json").write_text(json.dumps(meta, indent=1))
print("tersimpan:", sorted(f.name for f in folder.iterdir()))

koef = pd.Series(model[-1].coef_[0],
                 index=model[0].get_feature_names_out())
num = koef[koef.index.str.startswith("num__")]
num.index = num.index.str[5:]
print("\nkoefisien atribut numerik terbesar (dibakukan):")
for k in num.abs().sort_values(ascending=False).index[:6]:
    print(f"  {k:22s} {num[k]:+.2f}")

## Listing 27.3: Analisis uji acak dan kebijakan penyasaran

`kode/bab27/eksperimen.py`

In [ ]:
import sys; sys.modules.pop('siapkan', None); sys.path.insert(0, 'kode/bab27')
import joblib
import numpy as np
import pandas as pd
from siapkan import BIAYA, DATA, NILAI_TAHAN, fitur

model = joblib.load("kode/bab27/model/churn_v1.joblib")
d = pd.read_csv(DATA + "pelanggan_2026q3.csv")
d["p"] = model.predict_proba(fitur(d, "2026-09-30"))[:, 1]
v = d.kelompok == "voucer"

c = d.groupby("kelompok").churn.agg(["size", "mean"])
print(c.round(4).to_string())
beda = c.loc["kontrol", "mean"] - c.loc["voucer", "mean"]
se = np.sqrt(sum(c["mean"] * (1 - c["mean"]) / c["size"]))
print(f"efek voucer: churn turun {beda:.3f} "
      f"(95%: {beda - 1.96 * se:.3f} sampai {beda + 1.96 * se:.3f})")

d["desil"] = pd.qcut(d.p, 10, labels=range(1, 11))
t = d.pivot_table(index="desil", columns="kelompok", values="churn",
                  observed=True)
t["uplift"] = t.kontrol - t.voucer
t["laba_per_org"] = (t.uplift * NILAI_TAHAN - BIAYA).round(-2)
t["laba_per_org"] = t.laba_per_org.astype(int)
print("\nchurn per desil skor model:")
print(t.to_string(float_format="%.3f"))
t.assign(d=range(1, 11)).to_csv("gambar/bab27/uplift.dat", sep=" ",
                                index=False, float_format="%.4f")


def nilai_kebijakan(sasar, B=500, seed=0):
    """Laba per 1.000 anggota bila voucer dikirim ke anggota 'sasar',
    ditaksir dari uji acak (bootstrap untuk selang)."""
    rng = np.random.default_rng(seed)
    hasil = []
    for b in range(B + 1):
        i = (np.arange(len(d)) if b == 0 else
             rng.integers(0, len(d), len(d)))
        s, vv, y = sasar[i], v.to_numpy()[i], d.churn.to_numpy()[i]
        up = y[s & ~vv].mean() - y[s & vv].mean()
        hasil.append(s.mean() * 1000 * (up * NILAI_TAHAN - BIAYA))
    return hasil[0], np.percentile(hasil[1:], [2.5, 97.5])


peka = d.porsi_promo >= d.porsi_promo.median()
ambang = BIAYA / (0.3 * NILAI_TAHAN)
kebijakan = {"semua anggota": np.ones(len(d), bool),
             "20% skor tertinggi": (d.p >= d.p.quantile(.8))
             .to_numpy(),
             f"p >= {ambang:.3f} (Bab 7)": (d.p >= ambang).to_numpy(),
             "p >= ambang & peka promo": ((d.p >= ambang)
                                          & peka).to_numpy()}
print("\nlaba program per 1.000 anggota (ribu rupiah):")
for nama, s in kebijakan.items():
    m, (lo, hi) = nilai_kebijakan(s)
    print(f"  {nama:26s} sasaran {s.mean():5.1%}  {m / 1e3:8.0f}"
          f"  [{lo / 1e3:6.0f}, {hi / 1e3:6.0f}]")

## Listing 27.4: API skor churn dengan FastAPI

`kode/bab27/aplikasi/api.py`

Sel ini hanya menulis berkas aplikasi; jalankan aplikasinya dari terminal seperti dijelaskan di buku.

In [ ]:
%%writefile kode/bab27/aplikasi/api.py
"""Layanan skor churn: uvicorn kode.bab27.aplikasi.api:app"""
import json
from pathlib import Path
from typing import Literal
import joblib
import pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel, Field

FOLDER = Path(__file__).parents[1] / "model"
MODEL = joblib.load(FOLDER / "churn_v1.joblib")
META = json.loads((FOLDER / "churn_v1.json").read_text())
AMBANG = 0.278
app = FastAPI(title="Skor churn Kopi Nusantara", version=META["versi"])


class Anggota(BaseModel):
    id_anggota: str
    kota: str
    tingkat: Literal["perunggu", "perak", "emas"]
    jenis_kelamin: Literal["L", "P"]
    umur: int = Field(ge=17, le=90)
    frekuensi_90h: int = Field(ge=0)
    frekuensi_90h_lalu: int = Field(ge=0)
    recency_hari: int = Field(ge=0)
    nilai_90h: float = Field(ge=0)
    rata_nilai_struk: float = Field(ge=0)
    porsi_promo: float = Field(ge=0, le=1)
    porsi_aplikasi: float = Field(ge=0, le=1)
    porsi_kopi_hitam: float = Field(ge=0, le=1)
    porsi_pagi: float = Field(ge=0, le=1)
    porsi_akhir_pekan: float = Field(ge=0, le=1)
    ragam_menu: int = Field(ge=0)
    keluhan_90h: int = Field(ge=0)
    jarak_km: float = Field(ge=0)
    pakai_ulasan: Literal[0, 1]
    lama_anggota_bln: float = Field(ge=0)


@app.post("/skor")
def skor(daftar: list[Anggota]):
    X = pd.DataFrame([a.model_dump() for a in daftar])
    X["tren"] = X.frekuensi_90h / (X.frekuensi_90h_lalu + 1)
    p = MODEL.predict_proba(X[META["atribut"]])[:, 1]
    return {"versi_model": META["versi"],
            "hasil": [{"id_anggota": a.id_anggota,
                       "p_churn": round(float(q), 4),
                       "kirim_voucer": bool(q >= AMBANG)}
                      for a, q in zip(daftar, p)]}


@app.get("/sehat")
def sehat():
    return {"status": "ok", "versi_model": META["versi"],
            "auc_cv": META["auc_cv"]}

## Listing 27.5: Menguji API

`kode/bab27/uji_api.py`

In [ ]:
import json
import sys
import pandas as pd
from fastapi.testclient import TestClient
sys.path.insert(0, "kode/bab27/aplikasi")
from api import app

klien = TestClient(app)
sehat = klien.get("/sehat").json()
print("GET /sehat ->", sehat["status"], sehat["versi_model"],
      "AUC", sehat["auc_cv"])

d = pd.read_csv("kode/data/pelanggan_2026q3.csv").iloc[[0, 7]]
d["lama_anggota_bln"] = (pd.Timestamp("2026-09-30") - pd.to_datetime(
    d.tanggal_daftar)).dt.days / 30.44
kolom = list(app.openapi()["components"]["schemas"]["Anggota"]
             ["properties"])
r = klien.post("/skor", json=json.loads(d[kolom].to_json(
    orient="records")))
print("POST /skor ->", r.status_code)
for h in r.json()["hasil"]:
    print(f"  {h['id_anggota']}: p_churn {h['p_churn']}, "
          f"kirim_voucer {h['kirim_voucer']}")

salah = json.loads(d[kolom].iloc[:1].to_json(orient="records"))
salah[0]["porsi_promo"] = 1.7
r = klien.post("/skor", json=salah)
print("masukan salah ->", r.status_code,
      r.json()["detail"][0]["msg"])

## Listing 27.6: Dasbor retensi dengan Streamlit

`kode/bab27/aplikasi/dasbor.py`

Sel ini hanya menulis berkas aplikasi; jalankan aplikasinya dari terminal seperti dijelaskan di buku.

In [ ]:
%%writefile kode/bab27/aplikasi/dasbor.py
"""Dasbor retensi harian: streamlit run kode/bab27/aplikasi/dasbor.py"""
import pandas as pd
import streamlit as st

st.set_page_config(page_title="Retensi Kopi Nusantara", layout="wide")
skor = pd.read_parquet("keluaran/skor_harian.parquet")
pantau = pd.read_csv("keluaran/pantau_mingguan.csv")

st.title("Program retensi anggota")
a, b, c = st.columns(3)
a.metric("Anggota aktif", f"{len(skor):,}".replace(",", "."))
b.metric("Disasar minggu ini", int(skor.kirim_voucer.sum()))
c.metric("Rata-rata skor churn", f"{skor.p_churn.mean():.1%}")

kota = st.multiselect("Kota", sorted(skor.kota.unique()))
lihat = skor[skor.kota.isin(kota)] if kota else skor
st.bar_chart(lihat.groupby("kota").kirim_voucer.mean())
st.dataframe(lihat.nlargest(50, "p_churn")[
    ["id_anggota", "kota", "tingkat", "p_churn", "alasan_utama"]])

st.subheader("Kesehatan model")
st.line_chart(pantau.set_index("minggu")[["psi_skor", "auc_kontrol"]])
if pantau.psi_skor.iloc[-1] > 0.25:
    st.error("Sebaran skor bergeser besar: jadwalkan pelatihan ulang.")

## Listing 27.7: Memantau pergeseran dan kinerja

`kode/bab27/pantau.py`

In [ ]:
import sys; sys.modules.pop('siapkan', None); sys.path.insert(0, 'kode/bab27')
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from siapkan import DATA, NUMERIK, fitur

model = joblib.load("kode/bab27/model/churn_v1.joblib")
meta = json.load(open("kode/bab27/model/churn_v1.json"))
lama = fitur(pd.read_csv(DATA + "pelanggan.csv"), "2026-03-31")
d = pd.read_csv(DATA + "pelanggan_2026q3.csv")
baru = fitur(d, "2026-09-30")


def psi(a, b, k=10):
    """Population stability index: a = acuan, b = baru."""
    batas = np.unique(np.quantile(a, np.linspace(0, 1, k + 1)))
    batas[0], batas[-1] = -np.inf, np.inf
    pa = np.histogram(a, batas)[0] / len(a) + 1e-4
    pb = np.histogram(b, batas)[0] / len(b) + 1e-4
    return float(np.sum((pb - pa) * np.log(pb / pa)))


nilai = pd.Series({c: psi(lama[c], baru[c]) for c in NUMERIK})
print("PSI atribut (> 0,25 = bergeser besar, 0,1-0,25 = waspada):")
for c, v in nilai.sort_values(ascending=False)[:5].items():
    print(f"  {c:18s} {v:.3f}  rata-rata {lama[c].mean():8.2f}"
          f" -> {baru[c].mean():8.2f}")

p_lama = model.predict_proba(lama)[:, 1]
p_baru = model.predict_proba(baru)[:, 1]
print(f"\nPSI skor churn: {psi(p_lama, p_baru):.3f}; rata-rata skor "
      f"{p_lama.mean():.3f} -> {p_baru.mean():.3f}")

k = d.kelompok == "kontrol"
print(f"\nkinerja pada kelompok kontrol Q3 (n = {k.sum()}):")
print(f"  AUC {roc_auc_score(d.churn[k], p_baru[k]):.3f} "
      f"(saat dilatih {meta['auc_cv']:.3f})")
print(f"  churn nyata {d.churn[k].mean():.3f}, rata-rata tebakan "
      f"{p_baru[k].mean():.3f}")
print(f"  AUC bila kelompok voucer ikut dihitung: "
      f"{roc_auc_score(d.churn, p_baru):.3f}")

## Listing 27.8: Reservoir sampling dan statistik satu lintasan

`kode/bab27/aliran.py`

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(27)


def rp(x):
    return f"{x:,.0f}".replace(",", ".")


def reservoir(aliran, k, rng):
    """Sampel acak k butir dari aliran tanpa tahu panjangnya."""
    sampel = []
    for i, x in enumerate(aliran):
        if i < k:
            sampel.append(x)
        else:
            j = rng.integers(0, i + 1)
            if j < k:
                sampel[j] = x
    return sampel


item = pd.read_csv("kode/data/transaksi_lengkap_2026q1.csv",
                   usecols=["no_struk", "nilai"])
struk = item.groupby("no_struk").nilai.sum().rename("total")
struk = struk.reset_index()
contoh = reservoir(struk.itertuples(index=False), 2000, rng)
print(f"{rp(len(struk))} struk; rata-rata total sebenarnya",
      rp(struk.total.mean()))
print("rata-rata dari sampel reservoir 2.000 struk:",
      rp(np.mean([s.total for s in contoh])))

n, rata, m2 = 0, 0.0, 0.0
for x in struk.total:                    # algoritme Welford
    n += 1
    delta = x - rata
    rata += delta / n
    m2 += delta * (x - rata)
print(f"rata-rata dan sd satu lintasan: {rp(rata)} dan "
      f"{rp(np.sqrt(m2 / (n - 1)))} "
      f"(pandas: {rp(struk.total.std())})")